In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=32, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [4]:
from src.configs import S, B, C, IDX_TO_CLASS
from src.utils import convert_xywh_coords

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx].item()
                pred_class = IDX_TO_CLASS[pred_class_idx]
                
                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False, True)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False, True)
                
                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

In [5]:
X_batch, y_batch = next(iter(trainval_dl))

X_batch.shape, y_batch.shape

(torch.Size([32, 3, 224, 224]), torch.Size([32, 7, 7, 30]))

In [6]:
preds = model(X_batch)
preds.shape

torch.Size([32, 1470])

In [7]:
preds = preds.reshape((preds.shape[0], S, S, B * 5 + C))
preds.shape

torch.Size([32, 7, 7, 30])

In [8]:
decoded_preds = decode_preds(preds)
len(decoded_preds), len(decoded_preds[0])

(32, 98)

In [9]:
CONFIDENCE_THRESHOLD = 0.375
from operator import itemgetter

def filter_sort(decoded_preds):
    sorted_preds = []

    # 1. filter and  by class
    for image in decoded_preds:
        valid_preds = {}
        for pred in image:
            if pred[1] > CONFIDENCE_THRESHOLD:
                class_name = pred[0]
                if class_name in valid_preds:
                    valid_preds[class_name].append(pred)
                else:
                    valid_preds[class_name] = [pred]
    
        sorted_preds.append(valid_preds)

    # 2. sort each class by confidence score 
    for image in sorted_preds:
        for class_name in image:
            image[class_name].sort(key=itemgetter(1), reverse=True)
    
    return sorted_preds

In [10]:
sorted_preds = filter_sort(decoded_preds)
sorted_preds

[{},
 {'train': [('train',
    0.39393625904277485,
    120.8749771118164,
    -25.85848045349121,
    157.57272338867188,
    18.8292293548584)]},
 {},
 {},
 {'motorbike': [('motorbike',
    0.43030747432577954,
    164.83668518066406,
    49.54353713989258,
    101.44871520996094,
    108.13996887207031)],
  'horse': [('horse',
    0.3832108733558641,
    22.257619857788086,
    188.83712768554688,
    32.83544921875,
    201.4228515625)],
  'pottedplant': [('pottedplant',
    0.6724611486664145,
    41.29933166503906,
    202.91226196289062,
    83.35881042480469,
    152.251220703125)]},
 {'boat': [('boat',
    0.49704005969800846,
    101.05862426757812,
    90.35047912597656,
    12.536468505859375,
    49.7398681640625),
   ('boat',
    0.4737003149056349,
    149.9368896484375,
    83.01869201660156,
    199.31808471679688,
    145.6940460205078),
   ('boat',
    0.47072358658788005,
    145.62594604492188,
    104.31411743164062,
    80.44563293457031,
    129.2883758544922),


In [13]:
from src.postprocessing import NMS

preds_batch = model(X_batch)
final_preds = NMS(preds_batch)

TypeError: max() received an invalid combination of arguments - got (float, float), but expected one of:
 * (Tensor input, *, Tensor out = None)
 * (Tensor input, Tensor other, *, Tensor out = None)
 * (Tensor input, int dim, bool keepdim = False, *, tuple of Tensors out = None)
 * (Tensor input, name dim, bool keepdim = False, *, tuple of Tensors out = None)


In [ ]:
final_preds